This imports the pandas library and gives it the short name pd. Pandas is a Python library for working with tabular data (like Excel sheets, CSV files, datasets). Every time you use pandas functions, you’ll call them as pd.something(). 

The result is stored in the variable df.
df is a DataFrame = pandas’ main table object (rows & columns, like a spreadsheet).
df = your dataset as a table in memory.

Here you are converting columns to datetime format.
-   df["Charging Start Time"] selects the column named "Charging Start Time" from the table.
- pd.to_datetime(...) converts that column from strings (e.g. "2023-01-01 08:30:00") into proper datetime objects.
- After this:
    - You can do things like subtract start and end times,
    - Filter by dates,
    - Extract hour, day, month, etc.
    - Same for "Charging End Time".
So now both columns are treated as actual time data, not just text.

In [2]:
import pandas as pd

df = pd.read_csv("../data/raw/ev_charging_patterns.csv")

df["Charging Start Time"] = pd.to_datetime(df["Charging Start Time"])
df["Charging End Time"]   = pd.to_datetime(df["Charging End Time"])

# SHOW SUMMARY OF DATAFRAME (rows, column names, data types, how many non-null values per column)
df.info()

#Show first 5 rows of dataset
df.head()


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1320 entries, 0 to 1319
Data columns (total 20 columns):
 #   Column                                    Non-Null Count  Dtype         
---  ------                                    --------------  -----         
 0   User ID                                   1320 non-null   object        
 1   Vehicle Model                             1320 non-null   object        
 2   Battery Capacity (kWh)                    1320 non-null   float64       
 3   Charging Station ID                       1320 non-null   object        
 4   Charging Station Location                 1320 non-null   object        
 5   Charging Start Time                       1320 non-null   datetime64[ns]
 6   Charging End Time                         1320 non-null   datetime64[ns]
 7   Energy Consumed (kWh)                     1254 non-null   float64       
 8   Charging Duration (hours)                 1320 non-null   float64       
 9   Charging Rate (kW)            

,User ID,Vehicle Model,Battery Capacity (kWh),Charging Station ID,Charging Station Location,Charging Start Time,Charging End Time,Energy Consumed (kWh),Charging Duration (hours),Charging Rate (kW),Charging Cost (USD),Time of Day,Day of Week,State of Charge (Start %),State of Charge (End %),Distance Driven (since last charge) (km),Temperature (°C),Vehicle Age (years),Charger Type,User Type
0,User_1,BMW i3,108.463007,Station_391,Houston,2024-01-01 00:00:00,2024-01-01 00:39:00,60.712346,0.591363,36.389181,13.087717,Evening,Tuesday,29.371576,86.119962,293.602111,27.947953,2.0,DC Fast Charger,Commuter
1,User_2,Hyundai Kona,100.000000,Station_428,San Francisco,2024-01-01 01:00:00,2024-01-01 03:01:00,12.339275,3.133652,30.677735,21.128448,Morning,Monday,10.115778,84.664344,112.112804,14.311026,3.0,Level 1,Casual Driver
2,User_3,Chevy Bolt,75.000000,Station_181,San Francisco,2024-01-01 02:00:00,2024-01-01 04:48:00,19.128876,2.452653,27.513593,35.667270,Morning,Thursday,6.854604,69.917615,71.799253,21.002002,2.0,Level 2,Commuter
3,User_4,Hyundai Kona,50.000000,Station_327,Houston,2024-01-01 03:00:00,2024-01-01 06:42:00,79.457824,1.266431,32.882870,13.036239,Evening,Saturday,83.120003,99.624328,199.577785,38.316313,1.0,Level 1,Long-Distance Traveler
4,User_5,Hyundai Kona,50.000000,Station_108,Los Angeles,2024-01-01 04:00:00,2024-01-01 05:46:00,19.629104,2.019765,10.215712,10.161471,Morning,Saturday,54.258950,63.743786,203.661847,-7.834199,1.0,Level 1,Long-Distance Traveler


In [4]:
# Create a table of True/False df.isna()
# .sum() counts how many true values are present per column
df.isna().sum()

User ID                                      0
Vehicle Model                                0
Battery Capacity (kWh)                       0
Charging Station ID                          0
Charging Station Location                    0
Charging Start Time                          0
Charging End Time                            0
Energy Consumed (kWh)                       66
Charging Duration (hours)                    0
Charging Rate (kW)                          66
Charging Cost (USD)                          0
Time of Day                                  0
Day of Week                                  0
State of Charge (Start %)                    0
State of Charge (End %)                      0
Distance Driven (since last charge) (km)    66
Temperature (°C)                             0
Vehicle Age (years)                          0
Charger Type                                 0
User Type                                    0
dtype: int64

1) Goal: Ensure our target variable for modelling is always present.

dropna(subset=["Energy Consumed (kWh)"]) removes all rows where
"Energy Consumed (kWh)" is missing (NaN).

This column is the target for our regression model, so rows without it
cannot be used for supervised learning.

.copy() creates a new, independent DataFrame to avoid hidden links to
the old one and to prevent SettingWithCopyWarning when we modify df later.

2) Goal: Handle missing values in an input feature without discarding rows.

"Distance Driven (since last charge) (km)" is a feature, not the target.

df[...].median() computes the median distance over all non-missing rows.

.fillna(median_value) replaces all NaN values in that column with this
median.

Using the median is robust to outliers and allows us to keep those rows in
the dataset for training.

3) Goal: Visually compare the original duration with the recalculated one.

Shows the first few rows of:

"Charging Duration (hours)" (given by the dataset), and

"duration_calc_h" (computed from timestamps).

If these values are close, it confirms that the timestamps and the provided
duration are consistent. Large differences would indicate data quality issues.

In [5]:
df = df.dropna(subset=["Energy Consumed (kWh)"]).copy()

df["Distance Driven (since last charge) (km)"] = (
    df["Distance Driven (since last charge) (km)"]
    .fillna(df["Distance Driven (since last charge) (km)"].median())
)

df["duration_calc_h"] = (
    (df["Charging End Time"] - df["Charging Start Time"])
    .dt.total_seconds() / 3600
)

# Compare with existing duration:
df[["Charging Duration (hours)", "duration_calc_h"]].head()

,Charging Duration (hours),duration_calc_h
0,0.591363,0.650000
1,3.133652,2.016667
2,2.452653,2.800000
3,1.266431,3.700000
4,2.019765,1.766667


Step 2: Explanatory Data Analysis (EDA)
1. Univariate distributions 
    - Energy Consumed (kWh) 
    - Charging Duration (hours)
    - Charging Cost (USD)
    - Temperature (°C)
    - Distance Driven (since last charge) (km)

2. Time patterns 
    - 